In [14]:
import requests
import pandas as pd
import numpy as np
import os
import time
from datetime import datetime, timedelta
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, classification_report, confusion_matrix
)
from xgboost import XGBRegressor
from dotenv import load_dotenv

In [15]:
load_dotenv()

API_KEY = os.getenv("API_KEY")

In [16]:
def cached_download(ticker, cache_dir="stock_cache", retries=5, days=365):
    os.makedirs(cache_dir, exist_ok=True)
    safe_name = ticker.replace(":", "_").replace("/", "_")
    cache_file = f"{cache_dir}/{safe_name}.csv"

    # Return cached version if available
    if os.path.exists(cache_file):
        print(f"Loading {ticker} from cache")
        return pd.read_csv(cache_file, index_col=0, parse_dates=True)

    # Default date range
    end_date = datetime.today().strftime("%Y-%m-%d")
    start_date = (datetime.today() - timedelta(days=days)).strftime("%Y-%m-%d")

    url = f"https://api.polygon.io/v2/aggs/ticker/{ticker}/range/1/day/{start_date}/{end_date}"
    params = {
        "adjusted": "true",
        "sort": "asc",
        "limit": 50000,
        "apiKey": API_KEY
    }

    for attempt in range(retries):
        try:
            r = requests.get(url, params=params)

            # Handle HTTP level errors
            if r.status_code == 403:
                raise ValueError("Invalid API key")
            if r.status_code == 429:
                raise ConnectionError("Rate limited")
            r.raise_for_status()

            data = r.json()

            # Handle Polygon response status
            if data.get("status") == "ERROR":
                raise ValueError(f"Polygon error: {data.get('error', 'Unknown error')}")
            if not data.get("results"):
                raise ValueError(f"No data returned for {ticker}")

            # Parse into a clean DataFrame
            df = pd.DataFrame(data["results"])
            df = df.rename(columns={
                "t": "timestamp",
                "o": "open",
                "h": "high",
                "l": "low",
                "c": "close",
                "v": "volume",
                "vw": "vwap"
            })
            df["date"] = pd.to_datetime(df["timestamp"], unit="ms")
            df = df.set_index("date")
            # Indices (e.g. I:VIX) do not include volume or vwap - select only present columns
            preferred = ["open", "high", "low", "close", "volume", "vwap"]
            df = df[[c for c in preferred if c in df.columns]]

            # Cache and return
            df.to_csv(cache_file, index=True)
            print(f"Downloaded and cached {ticker}")
            return df

        except ValueError as e:
            # Don't retry on bad ticker or API key
            if "Invalid API key" in str(e) or "Polygon error" in str(e):
                raise
            wait = 2 ** attempt * 5
            print(f"Attempt {attempt + 1}/{retries} failed: {e}. Waiting {wait}s...")
            time.sleep(wait)

        except (ConnectionError, requests.exceptions.RequestException) as e:
            wait = 2 ** attempt * 5
            print(f"Attempt {attempt + 1}/{retries} - network error: {e}. Waiting {wait}s...")
            time.sleep(wait)

    raise Exception(f"Failed to download {ticker} after {retries} retries")

In [17]:
def engineer_features(df, vix_df=None):
    df = df.copy()

    # Price-based features
    df["daily_return"]   = df["close"].pct_change()
    df["weekly_return"]  = df["close"].pct_change(periods=5)
    df["ma_20"]          = df["close"].rolling(window=20).mean()
    df["dist_from_ma20"] = (df["close"] - df["ma_20"]) / df["ma_20"]
    df["daily_range"]    = (df["high"] - df["low"]) / df["close"]

    # Volume-based features
    df["volume_change"]  = df["volume"].pct_change()
    df["vwap_dist"]      = (df["close"] - df["vwap"]) / df["vwap"]

    # Momentum / volatility features
    df["volatility_20"]  = df["daily_return"].rolling(window=20).std()

    delta = df["close"].diff()
    gain  = delta.clip(lower=0).rolling(window=14).mean()
    loss  = -delta.clip(upper=0).rolling(window=14).mean()
    df["rsi_14"] = 100 - (100 / (1 + gain / loss))

    df["bb_position"] = (
        (df["close"] - df["ma_20"]) /
        (df["close"].rolling(20).std() * 2)
    )

    # ── Extreme event indicator via VIX ──
    # VIX > 30 is a widely used threshold for elevated market fear/stress.
    # Normalize both indexes to date-only before aligning, then reindex to
    # produce a Series (not an Index) so Series methods like .gt() are available.
    if vix_df is not None:
        vix_daily = vix_df["close"].copy()
        vix_daily.index = vix_daily.index.normalize()
        aligned = vix_daily.reindex(df.index.normalize())
        aligned.index = df.index
        df["is_major_event"] = aligned.gt(30).fillna(False).astype(int)
    else:
        df["is_major_event"] = 0

    # ── Targets ──
    # Regression target: 5-day forward return (magnitude + direction over next week)
    df["target_return"] = df["close"].pct_change(5).shift(-5)

    # Classification target: did price go up tomorrow? (1 = yes, 0 = no)
    df["target_direction"] = (df["close"].shift(-1) > df["close"]).astype(int)

    # Drop rows with NaN (from rolling windows and shift)
    df.dropna(inplace=True)

    return df

In [18]:
FEATURES = [
    "daily_return", "weekly_return", "dist_from_ma20",
    "daily_range",  "volume_change", "vwap_dist",
    "volatility_20","rsi_14",        "bb_position",
    "is_major_event"
]

TRAIN_WINDOW = 63   # ~3 months
TEST_WINDOW  = 42   # ~2 months


def walk_forward_splits(df, train_window=TRAIN_WINDOW, test_window=TEST_WINDOW):
    """
    Generate (train_indices, test_indices) pairs using a fixed sliding window.

    Each fold advances by test_window days:
      Fold 1: Train [0  → 62],  Test [63  → 104]
      Fold 2: Train [42 → 104], Test [105 → 146]
      ...
    """
    splits = []
    n = len(df)
    start = 0
    while start + train_window + test_window <= n:
        train_idx = list(range(start, start + train_window))
        test_idx  = list(range(start + train_window, start + train_window + test_window))
        splits.append((train_idx, test_idx))
        start += test_window
    return splits


def prepare_data(df, target_col, train_idx, test_idx):
    X = df[FEATURES].values
    y = df[target_col].values

    X_train, y_train = X[train_idx], y[train_idx]
    X_test,  y_test  = X[test_idx],  y[test_idx]

    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    return X_train, X_test, y_train, y_test, scaler

In [19]:
def run_linear_regression(df):
    splits = walk_forward_splits(df)

    print("\n" + "="*55)
    print("LINEAR REGRESSION — 5-day forward return")
    print(f"Walk-forward: {len(splits)} folds  "
          f"(train={TRAIN_WINDOW}d, test={TEST_WINDOW}d)")
    print("="*55)

    fold_metrics = []
    all_preds    = []
    last_model   = None

    for fold, (train_idx, test_idx) in enumerate(splits, 1):
        X_train, X_test, y_train, y_test, _ = prepare_data(
            df, "target_return", train_idx, test_idx
        )

        model = LinearRegression()
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        fold_metrics.append({
            "fold": fold,
            "mae":  mean_absolute_error(y_test, preds),
            "rmse": np.sqrt(mean_squared_error(y_test, preds)),
            "r2":   r2_score(y_test, preds),
        })

        test_dates = df.index[test_idx]
        for date, actual, pred in zip(test_dates, y_test, preds):
            all_preds.append({"date": date, "actual": actual, "predicted": pred})

        last_model = model

    metrics_df = pd.DataFrame(fold_metrics)
    preds_df   = pd.DataFrame(all_preds)

    print(f"\n  {'Metric':<8}  {'Mean':>10}  {'Std':>10}")
    print(f"  {'-'*32}")
    for col, label in [("rmse", "RMSE"), ("mae", "MAE"), ("r2", "R²")]:
        print(f"  {label:<8}  {metrics_df[col].mean():>10.6f}  ± {metrics_df[col].std():>9.6f}")

    print("\n  Feature coefficients (last fold):")
    coef_df = pd.DataFrame({
        "feature":     FEATURES,
        "coefficient": last_model.coef_
    }).sort_values("coefficient", key=abs, ascending=False)
    print(coef_df.to_string(index=False))

    return last_model, metrics_df, preds_df


In [20]:
def run_logistic_regression(df):
    splits = walk_forward_splits(df)

    print("\n" + "="*55)
    print("LOGISTIC REGRESSION — Predicting price direction")
    print(f"Walk-forward: {len(splits)} folds  "
          f"(train={TRAIN_WINDOW}d, test={TEST_WINDOW}d)")
    print("="*55)

    fold_metrics = []
    all_preds    = []
    agg_cm       = np.zeros((2, 2), dtype=int)
    last_model   = None

    for fold, (train_idx, test_idx) in enumerate(splits, 1):
        X_train, X_test, y_train, y_test, _ = prepare_data(
            df, "target_direction", train_idx, test_idx
        )

        model = LogisticRegression(max_iter=1000, C=0.1)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        report = classification_report(y_test, preds, output_dict=True, zero_division=0)
        fold_metrics.append({
            "fold":      fold,
            "accuracy":  accuracy_score(y_test, preds),
            "precision": report["macro avg"]["precision"],
            "recall":    report["macro avg"]["recall"],
            "f1":        report["macro avg"]["f1-score"],
        })
        agg_cm += confusion_matrix(y_test, preds)

        test_dates = df.index[test_idx]
        for date, actual, pred in zip(test_dates, y_test, preds):
            all_preds.append({"date": date, "actual": actual, "predicted": pred})

        last_model = model

    metrics_df = pd.DataFrame(fold_metrics)
    preds_df   = pd.DataFrame(all_preds)

    print(f"\n  {'Metric':<12}  {'Mean':>8}  {'Std':>8}")
    print(f"  {'-'*32}")
    for col, label in [("f1", "F1"), ("precision", "Precision"), ("recall", "Recall"), ("accuracy", "Accuracy")]:
        print(f"  {label:<12}  {metrics_df[col].mean():>8.4f}  ± {metrics_df[col].std():>7.4f}")

    print("\n  Aggregate confusion matrix (all folds):")
    cm_df = pd.DataFrame(agg_cm,
                         index=["Actual Down", "Actual Up"],
                         columns=["Pred Down",  "Pred Up"])
    print(cm_df.to_string())

    print("\n  Feature log-odds (last fold):")
    coef_df = pd.DataFrame({
        "feature":  FEATURES,
        "log_odds": last_model.coef_[0]
    }).sort_values("log_odds", key=abs, ascending=False)
    print(coef_df.to_string(index=False))

    return last_model, metrics_df, preds_df


In [21]:
def run_ridge_regression(df):
    splits = walk_forward_splits(df)

    print("\n" + "="*55)
    print("RIDGE REGRESSION — 5-day forward return")
    print(f"Walk-forward: {len(splits)} folds  "
          f"(train={TRAIN_WINDOW}d, test={TEST_WINDOW}d)")
    print("="*55)

    fold_metrics = []
    all_preds    = []
    last_model   = None

    for fold, (train_idx, test_idx) in enumerate(splits, 1):
        X_train, X_test, y_train, y_test, _ = prepare_data(
            df, "target_return", train_idx, test_idx
        )

        model = Ridge(alpha=1.0)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        fold_metrics.append({
            "fold": fold,
            "mae":  mean_absolute_error(y_test, preds),
            "rmse": np.sqrt(mean_squared_error(y_test, preds)),
            "r2":   r2_score(y_test, preds),
        })

        test_dates = df.index[test_idx]
        for date, actual, pred in zip(test_dates, y_test, preds):
            all_preds.append({"date": date, "actual": actual, "predicted": pred})

        last_model = model

    metrics_df = pd.DataFrame(fold_metrics)
    preds_df   = pd.DataFrame(all_preds)

    print(f"\n  {'Metric':<8}  {'Mean':>10}  {'Std':>10}")
    print(f"  {'-'*32}")
    for col, label in [("rmse", "RMSE"), ("mae", "MAE"), ("r2", "R²")]:
        print(f"  {label:<8}  {metrics_df[col].mean():>10.6f}  ± {metrics_df[col].std():>9.6f}")

    print("\n  Feature coefficients (last fold):")
    coef_df = pd.DataFrame({
        "feature":     FEATURES,
        "coefficient": last_model.coef_
    }).sort_values("coefficient", key=abs, ascending=False)
    print(coef_df.to_string(index=False))

    return last_model, metrics_df, preds_df


def run_xgboost(df):
    splits = walk_forward_splits(df)

    print("\n" + "="*55)
    print("XGBOOST — 5-day forward return")
    print(f"Walk-forward: {len(splits)} folds  "
          f"(train={TRAIN_WINDOW}d, test={TEST_WINDOW}d)")
    print("="*55)

    fold_metrics = []
    all_preds    = []
    last_model   = None

    for fold, (train_idx, test_idx) in enumerate(splits, 1):
        X_train, X_test, y_train, y_test, _ = prepare_data(
            df, "target_return", train_idx, test_idx
        )

        model = XGBRegressor(
            n_estimators=100,
            max_depth=3,
            learning_rate=0.1,
            random_state=42,
            verbosity=0
        )
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        fold_metrics.append({
            "fold": fold,
            "mae":  mean_absolute_error(y_test, preds),
            "rmse": np.sqrt(mean_squared_error(y_test, preds)),
            "r2":   r2_score(y_test, preds),
        })

        test_dates = df.index[test_idx]
        for date, actual, pred in zip(test_dates, y_test, preds):
            all_preds.append({"date": date, "actual": actual, "predicted": pred})

        last_model = model

    metrics_df = pd.DataFrame(fold_metrics)
    preds_df   = pd.DataFrame(all_preds)

    print(f"\n  {'Metric':<8}  {'Mean':>10}  {'Std':>10}")
    print(f"  {'-'*32}")
    for col, label in [("rmse", "RMSE"), ("mae", "MAE"), ("r2", "R²")]:
        print(f"  {label:<8}  {metrics_df[col].mean():>10.6f}  ± {metrics_df[col].std():>9.6f}")

    print("\n  Feature importances (last fold):")
    imp_df = pd.DataFrame({
        "feature":    FEATURES,
        "importance": last_model.feature_importances_
    }).sort_values("importance", ascending=False)
    print(imp_df.to_string(index=False))

    return last_model, metrics_df, preds_df


def run_random_forest(df):
    splits = walk_forward_splits(df)

    print("\n" + "="*55)
    print("RANDOM FOREST — Predicting price direction")
    print(f"Walk-forward: {len(splits)} folds  "
          f"(train={TRAIN_WINDOW}d, test={TEST_WINDOW}d)")
    print("="*55)

    fold_metrics = []
    all_preds    = []
    agg_cm       = np.zeros((2, 2), dtype=int)
    last_model   = None

    for fold, (train_idx, test_idx) in enumerate(splits, 1):
        X_train, X_test, y_train, y_test, _ = prepare_data(
            df, "target_direction", train_idx, test_idx
        )

        model = RandomForestClassifier(n_estimators=100, random_state=42)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        report = classification_report(y_test, preds, output_dict=True, zero_division=0)
        fold_metrics.append({
            "fold":      fold,
            "accuracy":  accuracy_score(y_test, preds),
            "precision": report["macro avg"]["precision"],
            "recall":    report["macro avg"]["recall"],
            "f1":        report["macro avg"]["f1-score"],
        })
        agg_cm += confusion_matrix(y_test, preds)

        test_dates = df.index[test_idx]
        for date, actual, pred in zip(test_dates, y_test, preds):
            all_preds.append({"date": date, "actual": actual, "predicted": pred})

        last_model = model

    metrics_df = pd.DataFrame(fold_metrics)
    preds_df   = pd.DataFrame(all_preds)

    print(f"\n  {'Metric':<12}  {'Mean':>8}  {'Std':>8}")
    print(f"  {'-'*32}")
    for col, label in [("f1", "F1"), ("precision", "Precision"), ("recall", "Recall"), ("accuracy", "Accuracy")]:
        print(f"  {label:<12}  {metrics_df[col].mean():>8.4f}  ± {metrics_df[col].std():>7.4f}")

    print("\n  Aggregate confusion matrix (all folds):")
    cm_df = pd.DataFrame(agg_cm,
                         index=["Actual Down", "Actual Up"],
                         columns=["Pred Down",  "Pred Up"])
    print(cm_df.to_string())

    print("\n  Feature importances (last fold):")
    imp_df = pd.DataFrame({
        "feature":    FEATURES,
        "importance": last_model.feature_importances_
    }).sort_values("importance", ascending=False)
    print(imp_df.to_string(index=False))

    return last_model, metrics_df, preds_df


In [22]:
SEASON_MAP = {
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Fall",  10: "Fall",  11: "Fall",
    12: "Winter", 1: "Winter", 2: "Winter"
}


def seasonal_summary(preds_df, task, label=""):
    """
    Break down model predictions by season and report per-season metrics.

    Parameters
    ----------
    preds_df : DataFrame with columns [date, actual, predicted]
    task     : "regression" or "classification"
    label    : display label for the header (e.g. "Linear Regression — SPY")
    """
    df = preds_df.copy()
    df["season"] = pd.to_datetime(df["date"]).dt.month.map(SEASON_MAP)

    header = f"SEASONAL BREAKDOWN — {label}" if label else "SEASONAL BREAKDOWN"
    print("\n" + "="*55)
    print(header)
    print("="*55)

    season_order = ["Spring", "Summer", "Fall", "Winter"]
    rows = []

    if task == "regression":
        for season in season_order:
            s = df[df["season"] == season]
            if len(s) < 2:
                continue
            rows.append({
                "season": season,
                "n":      len(s),
                "rmse":   round(np.sqrt(mean_squared_error(s["actual"], s["predicted"])), 6),
                "mae":    round(mean_absolute_error(s["actual"], s["predicted"]), 6),
                "r2":     round(r2_score(s["actual"], s["predicted"]), 4),
            })

    elif task == "classification":
        for season in season_order:
            s = df[df["season"] == season]
            if len(s) < 2:
                continue
            report = classification_report(
                s["actual"], s["predicted"], output_dict=True, zero_division=0
            )
            rows.append({
                "season":    season,
                "n":         len(s),
                "f1":        round(report["macro avg"]["f1-score"], 4),
                "precision": round(report["macro avg"]["precision"], 4),
                "recall":    round(report["macro avg"]["recall"], 4),
                "accuracy":  round(accuracy_score(s["actual"], s["predicted"]), 4),
            })

    summary = pd.DataFrame(rows)
    print(summary.to_string(index=False))
    return summary


In [23]:
SPY_TICKER   = "SPY"
TESLA_TICKER = "TSLA"
VIX_TICKER   = "I:VIX"
DAYS         = 730   # up to 2 years on free tier

spy_raw_df   = cached_download(SPY_TICKER,   days=DAYS)
tesla_raw_df = cached_download(TESLA_TICKER, days=DAYS)
vix_raw_df   = cached_download(VIX_TICKER,   days=DAYS)

Loading SPY from cache
Loading TSLA from cache
Loading I:VIX from cache


In [24]:
# SPY_TICKER   = "SPY"
# TESLA_TICKER = "TSLA"
VIX_TICKER   = "I:VIX"
DAYS         = 3650   # up to 2 years on free tier

# spy_raw_df   = cached_download(SPY_TICKER,   days=DAYS)
# tesla_raw_df = cached_download(TESLA_TICKER, days=DAYS)
vix_raw_df   = cached_download(VIX_TICKER,   days=DAYS)

Loading I:VIX from cache


In [25]:

len(vix_raw_df)

517

In [26]:
# Engineer features
spy_df = engineer_features(spy_raw_df, vix_df=vix_raw_df)
print(f"\nDataset: {len(spy_df)} rows after feature engineering")
print(f"Date range: {spy_df.index.min().date()} → {spy_df.index.max().date()}")
print(f"Major event days (VIX > 30): {spy_df['is_major_event'].sum()}")
print(f"\nFeature preview:\n{spy_df[FEATURES].head()}")

# Regression models
lin_model,   lin_metrics,   lin_preds   = run_linear_regression(spy_df)
seasonal_summary(lin_preds, "regression", "Linear Regression — SPY")

ridge_model, ridge_metrics, ridge_preds = run_ridge_regression(spy_df)
seasonal_summary(ridge_preds, "regression", "Ridge Regression — SPY")

xgb_model,   xgb_metrics,   xgb_preds   = run_xgboost(spy_df)
seasonal_summary(xgb_preds, "regression", "XGBoost — SPY")

# Classification models
log_model, log_metrics, log_preds = run_logistic_regression(spy_df)
seasonal_summary(log_preds, "classification", "Logistic Regression — SPY")

rf_model,  rf_metrics,  rf_preds  = run_random_forest(spy_df)
seasonal_summary(rf_preds, "classification", "Random Forest — SPY")



Dataset: 476 rows after feature engineering
Date range: 2024-05-08 → 2026-04-01
Major event days (VIX > 30): 15

Feature preview:
                     daily_return  weekly_return  dist_from_ma20  daily_range  \
date                                                                            
2024-05-08 04:00:00      0.000097       0.033656        0.020682     0.005027   
2024-05-09 04:00:00      0.005762       0.029978        0.026344     0.006733   
2024-05-10 04:00:00      0.001288       0.018678        0.026654     0.005846   
2024-05-13 04:00:00      0.000134       0.008402        0.025129     0.005625   
2024-05-14 04:00:00      0.004588       0.011912        0.027833     0.006249   

                     volume_change  vwap_dist  volatility_20     rsi_14  \
date                                                                      
2024-05-08 04:00:00      -0.200035   0.000520       0.008808  67.624177   
2024-05-09 04:00:00       0.037968   0.002042       0.008740  75.651282   
2

,season,n,f1,precision,recall,accuracy
0,Spring,63,0.4439,0.4574,0.4583,0.4444
1,Summer,80,0.4617,0.4783,0.4789,0.4625
2,Fall,126,0.5553,0.5562,0.5591,0.5714
3,Winter,109,0.5775,0.6077,0.5910,0.5963


In [27]:
# Engineer features
tesla_df = engineer_features(tesla_raw_df, vix_df=vix_raw_df)
print(f"\nDataset: {len(tesla_df)} rows after feature engineering")
print(f"Date range: {tesla_df.index.min().date()} → {tesla_df.index.max().date()}")
print(f"Major event days (VIX > 30): {tesla_df['is_major_event'].sum()}")
print(f"\nFeature preview:\n{tesla_df[FEATURES].head()}")

# Regression models
lin_model,   lin_metrics,   lin_preds   = run_linear_regression(tesla_df)
seasonal_summary(lin_preds, "regression", "Linear Regression — TSLA")

ridge_model, ridge_metrics, ridge_preds = run_ridge_regression(tesla_df)
seasonal_summary(ridge_preds, "regression", "Ridge Regression — TSLA")

xgb_model,   xgb_metrics,   xgb_preds   = run_xgboost(tesla_df)
seasonal_summary(xgb_preds, "regression", "XGBoost — TSLA")

# Classification models
log_model, log_metrics, log_preds = run_logistic_regression(tesla_df)
seasonal_summary(log_preds, "classification", "Logistic Regression — TSLA")

rf_model,  rf_metrics,  rf_preds  = run_random_forest(tesla_df)
seasonal_summary(rf_preds, "classification", "Random Forest — TSLA")



Dataset: 476 rows after feature engineering
Date range: 2024-05-08 → 2026-04-01
Major event days (VIX > 30): 15

Feature preview:
                     daily_return  weekly_return  dist_from_ma20  daily_range  \
date                                                                            
2024-05-08 04:00:00     -0.017378      -0.029279        0.040059     0.033826   
2024-05-09 04:00:00     -0.015739      -0.044664        0.024491     0.024714   
2024-05-10 04:00:00     -0.020352      -0.070203        0.004412     0.031518   
2024-05-13 04:00:00      0.020300      -0.069658        0.021631     0.037233   
2024-05-14 04:00:00      0.032928      -0.001462        0.048900     0.030527   

                     volume_change  vwap_dist  volatility_20     rsi_14  \
date                                                                      
2024-05-08 04:00:00       0.065608   0.005237       0.053220  63.395655   
2024-05-09 04:00:00      -0.175307  -0.006050       0.053250  63.484848   
2

,season,n,f1,precision,recall,accuracy
0,Spring,63,0.4714,0.4808,0.4818,0.4762
1,Summer,80,0.4972,0.5000,0.5000,0.5000
2,Fall,126,0.5453,0.5466,0.5474,0.5476
3,Winter,109,0.4771,0.4839,0.4839,0.4771
